In [ ]:
import pandas as pd
import plotly.express as px

config = {
    "toImageButtonOptions": {
        "format": "png",  # Ensure the format is PNG
        "filename": "high_res_plot",
        "height": 500,
        "width": 700,
        "scale": 3,  # Multiplies resolution/DPI (e.g., 2 or 3 for crisp images)
    }
}



# Figures for the paper

## Load the discovery results

In [ ]:
df = pd.read_csv("outputs/2.0-discovery_results.csv")

## Figure 3: Cumulative observation coverage

In [ ]:
field_count = (
    df.groupby("metadata_field")
    .agg(count=("snapshot_id", "nunique"))
    .sort_values("count", ascending=False)
)


In [ ]:
field_count["k"] = range(1, len(field_count) + 1)
field_count["coverage"] = field_count["count"].cumsum()


In [ ]:
field_count["coverage_pct"] = 100.0 * field_count["coverage"] / 3041

# 1. Create your base line graph
fig = px.line(
    field_count,
    x="k",
    y="coverage_pct",
    width=600,
    height=400,
    title="Cumulative observation coverage",
    labels={
        "coverage_pct": "Observations covered (%)",
        "k": "# of discovered fields",
    },
)

xx = 20
yy = field_count.loc[field_count["k"] == 20, "coverage_pct"].values[0]

# 2. Add vertical dotted line segment (from y=0 to y=50.81)
fig.add_shape(
    type="line",
    x0=xx, y0=0,
    x1=xx, y1=yy,
    line=dict(dash="dot", color="gray", width=1.5)
)

# 3. Add horizontal dotted line segment (from x=0 to x=20)
fig.add_shape(
    type="line",
    x0=0, y0=yy,
    x1=xx, y1=yy,
    line=dict(dash="dot", color="gray", width=1.5)
)

# 4. Add the tall, wrapped text box annotation
fig.add_annotation(
    x=xx,
    y=yy,
    # <br> tags break the text to make the box narrower and taller
    text=f"Top 20 fields<br>account for<br>{yy:.2f}% of<br>observations",
    align="center", # Centers the wrapped text inside the box
    showarrow=True,
    arrowhead=2,
    ax=80,    # Shifts text box 50 pixels right
    ay=30,   # Shifts text box 50 pixels up
    bordercolor="black",
    borderwidth=1,
    borderpad=6, # Adds padding inside the box to frame the wrapped text
    bgcolor="white",
    opacity=0.9
)

fig.show(config=config)

## Figure 4: Vocabulary accumulation curve

In [ ]:
acc_fields = set()
acc_snapshot = set()
snapshots = df["snapshot_id"].unique().tolist()
# np.random.shuffle(snapshots)

acc_curve = {}  # n_unique_snapshots: n_unique_fields
for x in snapshots:
    acc_snapshot.add(x)
    acc_fields.update(df[df["snapshot_id"] == x]["metadata_field"].unique().tolist())

    acc_curve[len(acc_snapshot)] = len(acc_fields)

acc_curve_df = (
    pd.Series(acc_curve)
    .rename("unique_metadata_field_count")
    .to_frame()
    .reset_index(names="unique_snapshot_count")
)

In [ ]:
fig = px.line(
    acc_curve_df,
    x="unique_snapshot_count",
    y="unique_metadata_field_count",
    height=400,
    width=600,
    title="Vocabulary accumulation curve",
    labels={
        "unique_metadata_field_count": "Unique metadata field count",
        "unique_snapshot_count": "# of snapshots processed",
    },
)

fig.show(config=config)